In [11]:

import numpy as np
from keras.models import Sequential
from keras.layers import GlobalAveragePooling2D,Conv2D,MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator    
from keras.applications import MobileNetV2
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score

In [12]:
train="archive/train"
test="/kaggle/input/fer2013/test"

In [13]:
emotions=['angry', 'happy', 'sad', 'surprise', 'neutral']

In [14]:
import tensorflow as tf


size=(128,128)


train_gen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)


train_data = train_gen.flow_from_directory(
    train,
    target_size=size,
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    classes=emotions
)

# Validation data (20%)
val_data = train_gen.flow_from_directory(
    train,
    target_size=size,
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    classes=emotions
)

# Test data (separate unseen dataset)
test_gen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

test_data = test_gen.flow_from_directory(
    test,
    target_size=size,
    batch_size=32,
    class_mode='categorical',
    shuffle=False,
    classes=emotions
)


Found 19341 images belonging to 5 classes.
Found 4835 images belonging to 5 classes.
Found 0 images belonging to 5 classes.


In [15]:
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
inputs = Input(shape=(128, 128, 3))
base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inputs)

x = base.output
x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D(pool_size=(2,2))(x)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
outputs = Dense(5, activation='softmax')(x)

model = Model(inputs, outputs)

model.summary()

C:\Users\Jishna\AppData\Local\Temp\ipykernel_23624\3791187237.py:4: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inputs)


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 64, 64,    │        864 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 64, 64,    │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 64, 64,    │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 64, 64,    │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 64, 64,    │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 64, 64,    │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 64, 64,    │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 64, 64,    │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 65, 65,    │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 32, 32,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 32, 32,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 32, 32,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 32, 32,    │      2,304 │ block_1_depthwis

 Total params: 3,087,109 (11.78 MB)

 Trainable params: 3,052,613 (11.64 MB)

 Non-trainable params: 34,496 (134.75 KB)

In [16]:
from keras.callbacks import ReduceLROnPlateau
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

In [17]:

model.fit(
    train_data,
    epochs=40,
    validation_data=val_data,
    callbacks=[early_stopping, reduce_lr]
)

Epoch 1/40
605/605 ━━━━━━━━━━━━━━━━━━━━ 1457s 2s/step - accuracy: 0.5121 - loss: 1.3162 - val_accuracy: 0.4014 - val_loss: 1.7776 - learning_rate: 0.0010
Epoch 2/40
605/605 ━━━━━━━━━━━━━━━━━━━━ 1521s 3s/step - accuracy: 0.6157 - loss: 1.1269 - val_accuracy: 0.5533 - val_loss: 1.4527 - learning_rate: 0.0010
Epoch 3/40
605/605 ━━━━━━━━━━━━━━━━━━━━ 1430s 2s/step - accuracy: 0.6459 - loss: 1.0699 - val_accuracy: 0.4366 - val_loss: 1.8856 - learning_rate: 0.0010
Epoch 4/40
605/605 ━━━━━━━━━━━━━━━━━━━━ 1393s 2s/step - accuracy: 0.6588 - loss: 1.0485 - val_accuracy: 0.3752 - val_loss: 1.7860 - learning_rate: 0.0010
Epoch 5/40
605/605 ━━━━━━━━━━━━━━━━━━━━ 6516s 11s/step - accuracy: 0.7070 - loss: 0.9626 - val_accuracy: 0.6587 - val_loss: 1.0796 - learning_rate: 2.0000e-04
Epoch 6/40
605/605 ━━━━━━━━━━━━━━━━━━━━ 3193s 5s/step - accuracy: 0.7202 - loss: 0.9343 - val_accuracy: 0.6821 - val_loss: 1.0077 - learning_rate: 2.0000e-04
Epoch 7/40
605/605 ━━━━━━━━━━━━━━━━━━━━ 2063s 3s/step - accuracy: 0

KeyboardInterrupt: 

In [ ]:
pred2=model.predict(test_data)

189/189 ━━━━━━━━━━━━━━━━━━━━ 35s 168ms/step


In [ ]:

print(accuracy_score(test_data.labels, np.argmax(pred2, axis=1)))

0.7565778586794638


In [ ]:
from sklearn.metrics import classification_report 
print(classification_report(test_data.classes,np.argmax(pred2, axis=1)))

              precision    recall  f1-score   support

           0       0.69      0.67      0.68       958
           1       0.91      0.87      0.89      1774
           2       0.65      0.67      0.66      1247
           3       0.85      0.84      0.85       831
           4       0.65      0.70      0.67      1233

    accuracy                           0.76      6043
   macro avg       0.75      0.75      0.75      6043
weighted avg       0.76      0.76      0.76      6043



In [18]:
model.save('mobilenetv2.keras')